In [ ]:
!pip install onnxruntime

In [3]:
!pip install tqdm

In [19]:
!pip uninstall tensorrt -y

Found existing installation: tensorrt 11.0.0.114
Uninstalling tensorrt-11.0.0.114:
  Successfully uninstalled tensorrt-11.0.0.114


In [22]:
!pip install tensorrt==8.6 --extra-index-url https://pypi.ngc.nvidia.com/

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com/
  Using cached tensorrt-8.6.0-cp310-none-manylinux_2_17_x86_64.whl.metadata (719 bytes)
Using cached tensorrt-8.6.0-cp310-none-manylinux_2_17_x86_64.whl (819.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 14.6 MB/s  0:00:39m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 14.4 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 15.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 MB 11.9 MB/s  0:00:58m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [tensorrt]4/5 [tensorrt]dnn-cu12]]u12]


In [3]:
conda install -c nvidia cudnn=8.9.7.29 -y


2 channel Terms of Service accepted
Channels:
 - nvidia
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /home/tachkin/miniconda3/envs/trt

  added / updated specs:
    - cudnn=8.9.7.29


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    cuda-nvrtc-12.9.86         |                0        64.1 MB  nvidia
    cuda-version-12.9          |                3          17 KB  nvidia
    libcublas-12.9.2.10        |       h46aabaf_0       446.2 MB  nvidia
    libstdcxx-15.2.0           |      h934c35e_19         5.6 MB  conda-forge
    libstdcxx-ng-15.2.0        |      hdf11a46_19          27 KB  conda-forge
    ----------------------------------

In [1]:
import tensorrt as trt
print(f'TensorRT: {trt.__version__}')

AttributeError: module 'tensorrt' has no attribute '__version__'

In [2]:
# benchmark_resnet50_inference.py
import torch
import torch.nn as nn
from torchvision import models, transforms
import time
import numpy as np
import onnx
import onnxruntime as ort
from tqdm import tqdm
import os

In [48]:
!pip install tensorrt==8.5.1.7


  Using cached tensorrt-8.5.1.7-cp310-none-manylinux_2_17_x86_64.whl.metadata (721 bytes)
Using cached tensorrt-8.5.1.7-cp310-none-manylinux_2_17_x86_64.whl (547.9 MB)
  Attempting uninstall: tensorrt
    Found existing installation: tensorrt 11.0.0.114
    Uninstalling tensorrt-11.0.0.114:
      Successfully uninstalled tensorrt-11.0.0.114


In [3]:
!pip install pycuda==2022.2.2


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 4.1 MB/s  0:00:00 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pycuda: filename=pycuda-2022.2.2-cp310-cp310-linux_x86_64.whl size=666375 sha256=3d5f57376a0831d110eacf4d0a1695a0838f5d5316707c187a8de05e598d8dae
  Stored in directory: /home/tachkin/.cache/pip/wheels/1d/7b/06/82a395a243fce00035dea9914d92bbef0013401497d849f8bc
Successfully built pycuda
  Attempting uninstall: pycuda
    Found existing installation: pycuda 2026.1
    Uninstalling pycuda-2026.1:
      Successfully uninstalled pycuda-2026.1


In [4]:
!pip install onnxruntime-gpu

In [3]:
# Конфигурация
BATCH_SIZES = [1, 32, 128, 256]
NUM_WARMUP_ITERATIONS = 5
NUM_BENCHMARK_ITERATIONS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Устройство: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA версия: {torch.version.cuda}")

Устройство: cuda
GPU: Tesla V100-SXM2-16GB
CUDA версия: 11.8


In [4]:
# Функция загрузки модели
def load_model(model_path=None, num_classes=10):
    """Загрузка модели ResNet-50 для CIFAR-10"""
    model = models.resnet50(pretrained=False, num_classes=num_classes)
    if model_path and os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        print(f"Загружены веса из {model_path}")
    else:
        print("Веса не найдены, используется случайная инициализация")
    model = model.to(DEVICE)
    model.eval()
    return model

In [5]:
# Точный таймер для GPU
class CUDATimer:
    def __init__(self):
        self.reset()
    
    def reset(self):
        if DEVICE.type == "cuda":
            self.start_event = torch.cuda.Event(enable_timing=True)
            self.end_event = torch.cuda.Event(enable_timing=True)
            self.start_event.record()
        else:
            self.start_time = time.time()
    
    def elapsed(self):
        if DEVICE.type == "cuda":
            self.end_event.record()
            torch.cuda.synchronize()
            return self.start_event.elapsed_time(self.end_event) / 1000.0
        return time.time() - self.start_time

In [6]:
model_path = "resnet50_cifar10_baseline.pth"
model = load_model(model_path, num_classes=10)

/home/tachkin/miniconda3/envs/torch_v100_pip/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/tachkin/miniconda3/envs/torch_v100_pip/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Загружены веса из resnet50_cifar10_baseline.pth


In [7]:
onnx_path = "model.onnx"
dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)
    
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    },
    opset_version=14
)

In [8]:
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

In [9]:
torch.__version__

'2.7.1+cu118'

In [11]:
import torchvision

In [12]:
torchvision.__version__

'0.22.1+cu118'

In [11]:
onnx.__version__

'1.22.0'

In [ ]:
import onnxruntime as ort


In [12]:
ort.__version__

'1.23.2'

In [10]:
ort_session = ort.InferenceSession(onnx_path, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
# Бенчмарк
results = {}
for batch_size in BATCH_SIZES:
    dummy_input_np = np.random.randn(batch_size, 3, 224, 224).astype(np.float32)
        
    # Прогрев
    for _ in range(5):
        ort_session.run(None, {'input': dummy_input_np})
        
    # Замер
    timer = time.time()
    for _ in range(NUM_BENCHMARK_ITERATIONS):
        ort_session.run(None, {'input': dummy_input_np})
    elapsed = time.time() - timer
        
    fps = (NUM_BENCHMARK_ITERATIONS * batch_size) / elapsed
    results[batch_size] = {'fps': fps, 'latency_ms': (elapsed/NUM_BENCHMARK_ITERATIONS)*1000}
    print(f"  Batch Size {batch_size:3d}: {fps:7.2f} img/sec")

  Batch Size   1:  328.78 img/sec
  Batch Size  32: 1085.40 img/sec
  Batch Size 128: 1182.66 img/sec
  Batch Size 256: 1201.94 img/sec


In [15]:
import tensorrt as trt
#import pycuda.driver as cuda
#import pycuda.autoinit

In [16]:
print(f"TensorRT версия: {trt.get_builder_version()}")


AttributeError: module 'tensorrt' has no attribute 'get_builder_version'

In [17]:
print(dir(trt))

['APILanguage', 'ActivationType', 'AllocatorFlag', 'AttentionIOForm', 'AttentionNormalizationOp', 'BoundingBoxFormat', 'Builder', 'BuilderFlag', 'CausalMaskKind', 'CollectiveOperation', 'CumulativeOperation', 'DataType', 'DeviceType', 'DimensionOperation', 'Dims', 'Dims2', 'Dims3', 'Dims4', 'DimsExprs', 'DimsHW', 'DynamicPluginTensorDesc', 'ElementWiseOperation', 'EngineCapability', 'EngineInspector', 'EngineStat', 'ErrorCode', 'ErrorCodeTRT', 'ExecutionContextAllocationStrategy', 'FallbackString', 'FillOperation', 'GatherMode', 'HardwareCompatibilityLevel', 'IActivationLayer', 'IAssertionLayer', 'IAttention', 'IAttentionBoundaryLayer', 'IAttentionInputLayer', 'IAttentionOutputLayer', 'IBuilderConfig', 'ICastLayer', 'IConcatenationLayer', 'IConditionLayer', 'IConstantLayer', 'IConvolutionLayer', 'ICudaEngine', 'ICumulativeLayer', 'IDebugListener', 'IDeconvolutionLayer', 'IDequantizeLayer', 'IDimensionExpr', 'IDistCollectiveLayer', 'IDynamicQuantizeLayer', 'IEinsumLayer', 'IElementWiseL

In [ ]:
logger = trt.Logger(trt.Logger.WARNING)
def build_engine(onnx_path, engine_path=None):
    builder = trt.Builder(logger)
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    parser = trt.OnnxParser(network, logger)
    
    with open(onnx_path, 'rb') as f:
        if not parser.parse(f.read()):
            for error in range(parser.num_errors):
                print(parser.get_error(error))
            return None
    
    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)  # 1GB
    
    # Для FP16:
    # if builder.platform_has_fast_fp16:
    #     config.set_flag(trt.BuilderFlag.FP16)
    
    engine = builder.build_engine_with_config(network, config)
    
    if engine_path and engine:
        with open(engine_path, 'wb') as f:
            f.write(engine.serialize())
        print(f"✅ Движок сохранён: {engine_path}")
    
    return engine

# Использование:
engine = build_engine('model.onnx', 'model.engine')

[06/22/2026-17:02:18] [TRT] [E] createInferBuilder: Error Code 1: Internal Error (Unsupported SM: 0x700 In smVersionToString at /_src/runtime/gpu/cask/caskUtils.cpp:1104)


TypeError: pybind11::init(): factory function returned nullptr

In [36]:
!pip uninstall tensorrt -y

Found existing installation: tensorrt 10.16.1.11
Uninstalling tensorrt-10.16.1.11:
  Successfully uninstalled tensorrt-10.16.1.11


In [37]:
!pip install tensorrt==8.5.1.7

  Using cached tensorrt-8.5.1.7-cp310-none-manylinux_2_17_x86_64.whl.metadata (721 bytes)
Using cached tensorrt-8.5.1.7-cp310-none-manylinux_2_17_x86_64.whl (547.9 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch-tensorrt 2.12.1 requires tensorrt<10.17.0,>=10.16.1, but you have tensorrt 8.5.1.7 which is incompatible.


In [38]:
!pip uninstall torch-tensorrt -y

Found existing installation: torch_tensorrt 2.12.1
Uninstalling torch_tensorrt-2.12.1:
  Successfully uninstalled torch_tensorrt-2.12.1


In [39]:
!pip install torch-tensorrt==1.4.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 2.9 MB/s  0:00:06 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 9.9 MB/s  0:00:506m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 18.4 MB/s  0:00:18m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 19.8 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 17.7 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 11.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.1/557.1 MB 19.4 MB/s  0:00:29m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 16.4 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 MB 15.1 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.6/102.6 MB 15.3 MB/s  0:00:06m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.2/173.2 MB 17.9 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import pycuda

In [6]:
pycuda.__cached__

'/home/tachkin/miniconda3/envs/torch_v100_pip/lib/python3.10/site-packages/pycuda/__pycache__/__init__.cpython-310.pyc'